In [4]:
import pandas as pd

import sys
sys.path.append('../..')
# from regioselect.__init__ import app, db, regioselect_results

In [5]:
from flask import Flask, render_template, request, redirect, url_for, flash, jsonify, send_from_directory
from flask_sqlalchemy import SQLAlchemy
from sqlalchemy.sql import func
import os

app = Flask(__name__)
app.config.from_object(__name__)
app.config['SECRET_KEY'] = '975f4ba64117f4cc65044299552ee698'
# app.config['DEBUG'] = False

# Database name
db_name = 'data/regioselect.db'
basedir = '/Users/nicolairee/KU_data/regioselect/regioselect' #os.path.dirname(os.path.abspath(__file__))
db_path = os.path.join(basedir, db_name)

# Configure the SQLite database URI
app.config['SQLALCHEMY_DATABASE_URI'] = 'sqlite:///' + db_path
app.config['SQLALCHEMY_TRACK_MODIFICATIONS'] = False

# initialize the app with Flask-SQLAlchemy
db = SQLAlchemy(app)


class regioselect_results(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    hash_code = db.Column(db.String(64), unique=True, nullable=False)
    rdkit_smiles = db.Column(db.String(400), unique=True, nullable=False)
    
    # types of status: none (not submitted), error (calc failed e.g. due to timeout), pending (calc in queue or running), complete (calc done)
    # ml_status = db.Column(db.String(10), default='none')
    xtb_status = db.Column(db.String(10), default='none')
    dft_sp_status = db.Column(db.String(10), default='none')
    dft_opt_status = db.Column(db.String(10), default='none')

    created_at = db.Column(db.DateTime(timezone=True), server_default=func.now())
    
    def __repr__(self):
        return f'<Molecule: {self.rdkit_smiles} ({self.hash_code})>'

In [6]:
app.app_context().push()

In [7]:
import sqlite3

try:
    conn = sqlite3.connect(db_path)

    column = db.Column('ml_status', db.String(10), default='none')
    column_name = column.compile(dialect=db.engine.dialect)
    column_type = column.type.compile(db.engine.dialect)

    # sql = f"""ALTER TABLE regioselect_results ADD COLUMN {column_name} {column_type}"""
    sql = f"""ALTER TABLE regioselect_results ADD COLUMN {column_name} {column_type} DEFAULT none"""

    cursor = conn.cursor()
    cursor.execute(sql)
    conn.commit()
    print("Altering OK")
    conn.close()

except sqlite3.Error as e:
    print("Error while Altering Table", e)


finally:
    if (conn):
        conn.close()
        print("Connection Closed")

Error while Altering Table duplicate column name: ml_status
Connection Closed


In [8]:
app = Flask(__name__)
app.config.from_object(__name__)
app.config['SECRET_KEY'] = '975f4ba64117f4dc65044299552ee698'
# app.config['DEBUG'] = False

# Database name
db_name = 'data/regioselect.db'
basedir = '/Users/nicolairee/KU_data/regioselect/regioselect' #os.path.dirname(os.path.abspath(__file__))
db_path = os.path.join(basedir, db_name)

# Configure the SQLite database URI
app.config['SQLALCHEMY_DATABASE_URI'] = 'sqlite:///' + db_path
app.config['SQLALCHEMY_TRACK_MODIFICATIONS'] = False

# initialize the app with Flask-SQLAlchemy
db = SQLAlchemy(app)

class regioselect_results(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    hash_code = db.Column(db.String(64), unique=True, nullable=False)
    rdkit_smiles = db.Column(db.String(400), unique=True, nullable=False)
    
    # types of status: none (not submitted), error (calc failed e.g. due to timeout), pending (calc in queue or running), complete (calc done)
    ml_status = db.Column(db.String(10), default='none')
    xtb_status = db.Column(db.String(10), default='none')
    dft_sp_status = db.Column(db.String(10), default='none')
    dft_opt_status = db.Column(db.String(10), default='none')

    created_at = db.Column(db.DateTime(timezone=True), server_default=func.now())
    
    def __repr__(self):
        return f'<Molecule: {self.rdkit_smiles} ({self.hash_code})>'

In [17]:
app.app_context().push()

In [18]:
# db.drop_all()
# db.create_all()

In [19]:
regioselect_results.query.all()

[<Molecule: CCOC(=O)c1cc(Cl)n2nc(-c3ccc(Br)cc3F)cc2n1 (f302f6b8b26306c5e315a67a2346781a)>,
 <Molecule: Cc1ccoc1 (6f607b0c73aad9228144fe081c5682ed)>,
 <Molecule: N (8d9c307cb7f3c4a32822a51922d1ceaa)>,
 <Molecule: c1ccc2ccccc2c1 (82cce384d65aa7736f2fdbdd401d54c2)>,
 <Molecule: O=C(O)c1ccc2c(c1)OCCO2 (da23d62f5607784cf778cf46af1d474b)>,
 <Molecule: N#Cc1ccccc1 (a808b2f56c754af832312f59d437a7cb)>,
 <Molecule: CC(C(=O)O)c1ccc2c(c1)[nH]c1ccc(Cl)cc12 (f9f474f253d1f7984936e15b60919845)>,
 <Molecule: Cc1ccno1 (f83a8232267317f3ae49e7c1f8bb67c2)>]

In [20]:
prediction = regioselect_results.query.first()
prediction.created_at.strftime("%d %b %Y (UTC time: %H:%M:%S)")

'28 Oct 2024 (UTC time: 12:49:36)'

In [21]:
from collections import defaultdict
from sqlalchemy.inspection import inspect

def query_to_dict(rset):
    result = defaultdict(list)
    for obj in rset:
        instance = inspect(obj)
        for key, x in instance.attrs.items():
            result[key].append(x.value)
    return result

app.app_context().push()

# hash_code = 'c18263c911e71db08ef4e310a681acf8'
# df = pd.DataFrame(query_to_dict(regioselect_results.query.filter_by(hash_code=hash_code))).set_index('id')

df = pd.DataFrame(query_to_dict(regioselect_results.query.all())).set_index('id')
df

,hash_code,rdkit_smiles,ml_status,xtb_status,dft_sp_status,dft_opt_status,created_at
id,,,,,,,
1,f302f6b8b26306c5e315a67a2346781a,CCOC(=O)c1cc(Cl)n2nc(-c3ccc(Br)cc3F)cc2n1,complete,none,none,none,2024-10-28 12:49:36
2,6f607b0c73aad9228144fe081c5682ed,Cc1ccoc1,complete,none,none,none,2024-10-28 12:51:25
3,8d9c307cb7f3c4a32822a51922d1ceaa,N,complete,none,none,none,2024-10-28 12:57:50
4,82cce384d65aa7736f2fdbdd401d54c2,c1ccc2ccccc2c1,complete,none,none,none,2024-10-28 12:58:04
5,da23d62f5607784cf778cf46af1d474b,O=C(O)c1ccc2c(c1)OCCO2,complete,none,none,none,2024-10-28 12:58:35
6,a808b2f56c754af832312f59d437a7cb,N#Cc1ccccc1,complete,none,none,none,2024-10-28 13:04:02
7,f9f474f253d1f7984936e15b60919845,CC(C(=O)O)c1ccc2c(c1)[nH]c1ccc(Cl)cc12,complete,none,none,none,2024-10-28 13:04:31
8,f83a8232267317f3ae49e7c1f8bb67c2,Cc1ccno1,complete,none,none,none,2024-10-28 13:05:42


In [22]:
df[df['ml_status'] == 'complete']

,hash_code,rdkit_smiles,ml_status,xtb_status,dft_sp_status,dft_opt_status,created_at
id,,,,,,,
1,f302f6b8b26306c5e315a67a2346781a,CCOC(=O)c1cc(Cl)n2nc(-c3ccc(Br)cc3F)cc2n1,complete,none,none,none,2024-10-28 12:49:36
2,6f607b0c73aad9228144fe081c5682ed,Cc1ccoc1,complete,none,none,none,2024-10-28 12:51:25
3,8d9c307cb7f3c4a32822a51922d1ceaa,N,complete,none,none,none,2024-10-28 12:57:50
4,82cce384d65aa7736f2fdbdd401d54c2,c1ccc2ccccc2c1,complete,none,none,none,2024-10-28 12:58:04
5,da23d62f5607784cf778cf46af1d474b,O=C(O)c1ccc2c(c1)OCCO2,complete,none,none,none,2024-10-28 12:58:35
6,a808b2f56c754af832312f59d437a7cb,N#Cc1ccccc1,complete,none,none,none,2024-10-28 13:04:02
7,f9f474f253d1f7984936e15b60919845,CC(C(=O)O)c1ccc2c(c1)[nH]c1ccc(Cl)cc12,complete,none,none,none,2024-10-28 13:04:31
8,f83a8232267317f3ae49e7c1f8bb67c2,Cc1ccno1,complete,none,none,none,2024-10-28 13:05:42


In [23]:
for hash_code in df[df['ml_status'] == 'error'].hash_code.tolist():
    print(hash_code)
    obj = regioselect_results.query.filter_by(hash_code=hash_code).one()
    db.session.delete(obj)
    db.session.commit()

f302f6b8b26306c5e315a67a2346781a
6f607b0c73aad9228144fe081c5682ed
8d9c307cb7f3c4a32822a51922d1ceaa
82cce384d65aa7736f2fdbdd401d54c2
da23d62f5607784cf778cf46af1d474b
a808b2f56c754af832312f59d437a7cb
f9f474f253d1f7984936e15b60919845
f83a8232267317f3ae49e7c1f8bb67c2


In [25]:
db.session.close()

In [33]:
# Delete an element in list
obj = regioselect_results.query.filter_by(hash_code='fe7adf4423471f288aa695ddf1c64df0').one()
db.session.delete(obj)
db.session.commit()